In [1]:
import os
import numpy as np
import pandas as pd

BASE = "/kaggle/input/plant-pathology-2020-fgvc7"
if not os.path.exists(BASE):
    BASE = "/kaggle/data/plant-pathology-2020-fgvc7"

train_csv_path = os.path.join(BASE, "train.csv")
test_csv_path = os.path.join(BASE, "test.csv")
sample_sub_path = os.path.join(BASE, "sample_submission.csv")
images_dir = os.path.join(BASE, "images")

print("BASE:", BASE)
print("train exists:", os.path.exists(train_csv_path))
print("test exists:", os.path.exists(test_csv_path))
print("sample exists:", os.path.exists(sample_sub_path))
print("images_dir exists:", os.path.exists(images_dir))

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)
sample_sub = pd.read_csv(sample_sub_path)

target_cols = ["healthy", "multiple_diseases", "rust", "scab"]
assert all(c in train_df.columns for c in ["image_id"] + target_cols)
assert list(sample_sub.columns) == ["image_id"] + target_cols

train_df.head()




BASE: /kaggle/input/plant-pathology-2020-fgvc7
train exists: True
test exists: True
sample exists: True
images_dir exists: True


,image_id,healthy,multiple_diseases,rust,scab
0,Train_0,0,0,1,0
1,Train_1,1,0,0,0
2,Train_2,0,0,1,0
3,Train_3,1,0,0,0
4,Train_4,0,0,1,0


In [2]:
from PIL import Image
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.cross_decomposition import PLSRegression
from sklearn.base import BaseEstimator, TransformerMixin


def _find_image_path(image_id: str) -> str:
    candidates = [
        os.path.join(images_dir, f"{image_id}.jpg"),
        os.path.join(images_dir, f"{image_id}.JPG"),
        os.path.join(images_dir, f"{image_id}.jpeg"),
        os.path.join(images_dir, f"{image_id}.JPEG"),
        os.path.join(images_dir, f"{image_id}.png"),
        os.path.join(images_dir, f"{image_id}.PNG"),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"Image file not found for image_id={image_id}. Tried: {candidates}"
    )


def load_image_vector(image_id, size=(64, 64)):
    path = _find_image_path(image_id)
    with Image.open(path) as img:
        img = img.convert("RGB")
        img = img.resize(size, resample=Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float32) / 255.0  # (H,W,3)
    return arr.reshape(-1)


X_train = np.stack([load_image_vector(i) for i in train_df["image_id"].values], axis=0)
y_train = train_df[target_cols].astype(np.float32).values
X_test = np.stack([load_image_vector(i) for i in test_df["image_id"].values], axis=0)

print("X_train:", X_train.shape, "y_train:", y_train.shape, "X_test:", X_test.shape)




X_train: (1638, 12288) y_train: (1638, 4) X_test: (183, 12288)


In [3]:
def rowwise_standardize(X):
    """
    Score-relevant: keep per-image normalization (better than per-feature scaling for raw pixels).
    """
    X = np.asarray(X, dtype=np.float32)
    mu = X.mean(axis=1, keepdims=True)
    sig = X.std(axis=1, keepdims=True)
    sig = np.where(sig < 1e-6, 1.0, sig)
    return (X - mu) / sig


def signed_sqrt(X):
    """
    Score-relevant: simple deterministic power transform to improve separability post-normalization.
    """
    X = np.asarray(X, dtype=np.float32)
    return np.sign(X) * np.sqrt(np.abs(X) + 1e-6)


class PLSXExtractor(BaseEstimator, TransformerMixin):
    """
    Bugfix: In some sklearn versions/set_output configurations, PLSRegression inside a Pipeline
    can yield a tuple-like output (X_scores, Y_scores) or an ndarray shaped (2, n_samples, k).
    Downstream estimators require a 2D (n_samples, k) feature matrix, so we always extract
    and return only the X-side scores deterministically.
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, (tuple, list)):
            if len(X) < 1:
                raise ValueError("PLSXExtractor got empty tuple/list output.")
            X = X[0]

        X = np.asarray(X)

        if X.ndim == 3 and X.shape[0] == 2:
            X = X[0]

        while X.ndim > 2 and X.shape[-1] == 1:
            X = X[..., 0]

        if X.ndim != 2:
            raise ValueError(
                f"PLSXExtractor expected 2D after extraction, got shape {X.shape}"
            )

        return X


def _ovr_proba_to_2d(proba, n_classes):
    """
    Stability: OneVsRestClassifier.predict_proba may return:
    - ndarray of shape (n_samples, n_classes), or
    - list of arrays (each (n_samples, 2) or (n_samples,))
    Convert to a single (n_samples, n_classes) array of positive-class probs.
    """
    if isinstance(proba, list):
        cols = []
        for p in proba:
            p = np.asarray(p)
            if p.ndim == 2 and p.shape[1] == 2:
                cols.append(p[:, 1])
            elif p.ndim == 1:
                cols.append(p)
            else:
                raise ValueError(f"Unexpected proba element shape: {p.shape}")
        proba = np.stack(cols, axis=1)

    proba = np.asarray(proba, dtype=np.float32)
    if proba.ndim == 1:
        proba = proba.reshape(-1, 1)

    if proba.shape[1] != n_classes:
        raise ValueError(f"Got proba shape {proba.shape}, expected (*, {n_classes})")
    return proba


def predict_proba_robust(fitted_model, X, n_classes):
    """
    Stability: prefer true predict_proba from the fitted pipeline; only fall back if unavailable.
    Also normalize OVR outputs to a (n_samples, n_classes) float array.
    """
    if hasattr(fitted_model, "predict_proba"):
        proba = fitted_model.predict_proba(X)
        proba = _ovr_proba_to_2d(proba, n_classes)
    else:
        scores = fitted_model.decision_function(X)
        scores = np.asarray(scores, dtype=np.float32)
        if scores.ndim == 1:
            scores = scores.reshape(-1, 1)
        proba = 1.0 / (1.0 + np.exp(-scores))
        proba = _ovr_proba_to_2d(proba, n_classes)

    return proba


# Bugfix/stability: ensure n_components is within PLS' valid bounds for this problem:
# n_components <= min(n_samples-1, n_features, n_targets)
max_pls = int(min(X_train.shape[0] - 1, X_train.shape[1], y_train.shape[1]))
pls_n_components = int(min(96, max_pls))
pls_n_components = max(pls_n_components, 2)
print(
    "Using PLSRegression n_components =",
    pls_n_components,
    " (max possible:",
    max_pls,
    ")",
)

y_train_ovr = np.asarray(y_train, dtype=np.float32)  # (n, 4) 0/1

model = Pipeline(
    steps=[
        ("row_norm", FunctionTransformer(rowwise_standardize, validate=False)),
        ("ssqrt", FunctionTransformer(signed_sqrt, validate=False)),
        (
            "pls",
            # Bugfix: remove unsupported 'mode' kwarg (varies by sklearn version)
            PLSRegression(
                n_components=pls_n_components,
                scale=False,  # already normalized
                max_iter=1000,
                tol=1e-06,
            ),
        ),
        ("pls_x", PLSXExtractor()),
        (
            "clf",
            OneVsRestClassifier(
                LogisticRegression(
                    max_iter=3000,
                    solver="saga",
                    C=8.0,
                    penalty="l2",
                    n_jobs=-1,
                    random_state=42,
                )
            ),
        ),
    ]
)

model.fit(X_train, y_train_ovr)

proba = predict_proba_robust(model, X_test, n_classes=len(target_cols))
proba = np.asarray(proba, dtype=np.float32)

sub = test_df[["image_id"]].copy()
sub[target_cols] = proba
sub[target_cols] = sub[target_cols].clip(0.0, 1.0)

sub = sub[["image_id"] + target_cols]
sub.to_csv("submission.csv", index=False)

print(sub.head())
print("Wrote submission.csv with shape:", sub.shape)
print("Columns:", list(sub.columns))
print("submission.csv exists:", os.path.exists("submission.csv"))

Using PLSRegression n_components = 4  (max possible: 4 )
  image_id   healthy  multiple_diseases      rust      scab
0   Test_0  0.301614           0.030047  0.342652  0.268753
1   Test_1  0.244003           0.030926  0.351568  0.316386
2   Test_2  0.378076           0.043343  0.169965  0.382220
3   Test_3  0.456033           0.015368  0.274231  0.242868
4   Test_4  0.323678           0.048000  0.263701  0.310302
Wrote submission.csv with shape: (183, 5)
Columns: ['image_id', 'healthy', 'multiple_diseases', 'rust', 'scab']
submission.csv exists: True
